# Session Explorer

Load and visualize a sensor capture session from the Sensor Capture app.

**Sensors captured**: Accelerometer (~500 Hz), Gyroscope (~500 Hz), Magnetometer (~50 Hz), GPS (~1 Hz)

If no real data is available, a synthetic session with simulated pothole events is generated automatically.

**Works in Google Colab** — no local install needed. Open via:
```
https://colab.research.google.com/github/SebastienBinet/NidsDePoule/blob/data_analysis_tools/data_capture_tools/analysis/notebooks/01_explore_session.ipynb
```

In [ ]:
# Install dependencies (needed for Colab; harmless if already installed locally)
%pip install -q folium scipy pandas numpy matplotlib gdown
# Install capture_analysis package (loader + synthetic generator) from GitHub
%pip install -q 'capture-analysis @ git+https://github.com/sebastienbinet/nidsdepoule.git@data_analysis_tools#subdirectory=data_capture_tools/analysis'

In [ ]:
# =============================================================================
# NOTEBOOK VERSION — bump when the notebook is updated to require a newer
# capture app version (adds a column, changes a field, etc.)
# =============================================================================
NOTEBOOK_VERSION = 'v020f'
print(f'Notebook version: {NOTEBOOK_VERSION}')

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
import folium

plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['figure.dpi'] = 100


def show_time_on_all(axes):
    """Show time x-axis tick labels on every subplot even when sharex=True."""
    for ax in (axes if hasattr(axes, '__iter__') else [axes]):
        ax.tick_params(axis='x', labelbottom=True)


# --- Inline session loader (works in Colab without local package install) ---

def load_session(session_dir):
    """Load all CSV files and metadata from a capture session directory."""
    d = Path(session_dir)
    meta_files = list(d.glob("meta_*.json"))
    if not meta_files:
        raise FileNotFoundError(f"No meta_*.json found in {d}")
    meta = json.loads(meta_files[0].read_text())

    start_boot_ns = meta.get("start_boot_time_ns", 0)
    boot_offset_ms = meta.get("boot_to_epoch_offset_ms", 0)

    def _load_sensor(prefix):
        files = list(d.glob(f"{prefix}_*.csv"))
        if not files:
            return None
        df = pd.read_csv(files[0], comment="#")
        if "timestamp_ns" in df.columns:
            t0 = start_boot_ns if start_boot_ns > 0 else df["timestamp_ns"].iloc[0]
            df["time_s"] = (df["timestamp_ns"] - t0) / 1e9
        return df

    accel = _load_sensor("accel")
    lin_accel = _load_sensor("lin_accel")
    gyro = _load_sensor("gyro")
    mag = _load_sensor("mag")

    gps_files = list(d.glob("gps_*.csv"))
    gps = None
    if gps_files:
        gps = pd.read_csv(gps_files[0], comment="#")
        if "timestamp_ms" in gps.columns:
            start_epoch_ms = boot_offset_ms + start_boot_ns / 1e6
            gps["time_s"] = (gps["timestamp_ms"] - start_epoch_ms) / 1e3

    # Load events (ground-truth labels from BT button, screen tap, etc.)
    events_files = list(d.glob("events_*.csv"))
    events = None
    if events_files:
        events = pd.read_csv(events_files[0], comment="#")
        if "timestamp_ms" in events.columns:
            start_epoch_ms = boot_offset_ms + start_boot_ns / 1e6
            events["time_s"] = (events["timestamp_ms"] - start_epoch_ms) / 1e3

    return {"meta": meta, "accel": accel, "lin_accel": lin_accel, "gyro": gyro, "mag": mag, "gps": gps, "events": events}


def plot_event_markers(ax, events, t_start=None, t_end=None):
    """Draw vertical lines for labeled events on a matplotlib axis."""
    if events is None or len(events) == 0:
        return
    colors = {"pothole": "red", "crack": "orange", "rough": "brown", "other": "gray"}
    for _, ev in events.iterrows():
        t = ev["time_s"]
        if t_start is not None and (t < t_start or t > t_end):
            continue
        c = colors.get(ev.get("event_type", "other"), "gray")
        ax.axvline(t, color=c, linestyle="--", alpha=0.7, linewidth=1.2)


print('Ready.')

## Load session

### Setup (once)
1. Create a Google Sheet with two columns: `session_name` | `drive_link`
2. Publish it: **File → Share → Publish to web → CSV**
3. Paste the published URL below as `INDEX_URL`

### After each capture
1. Share the session zip from the app to Google Drive
2. In Drive on your phone: long-press the zip → **Copy link**
3. Open the Google Sheet → add a row: session name + link

### Without real data
Leave `INDEX_URL = None` to use synthetic demo data.

In [ ]:
# --- Google Sheet session index ---
# Paste your published Google Sheet CSV URL here (set up once, never changes)
INDEX_URL = 'https://docs.google.com/spreadsheets/d/e/2PACX-1vRsL1exVeheee2v7Stqvu0XFrw3EezoSIw3X-AEoD80-iyN24x8f9dHH1l90kycQMkQUPJcV5MU4RDB/pub?output=csv'

import os, zipfile, re

sessions_available = []

if INDEX_URL:
    index_df = pd.read_csv(INDEX_URL)
    print(f'Found {len(index_df)} session(s) in index:\n')
    for i, row in index_df.iterrows():
        print(f'  [{i}] {row.iloc[0]}')
        sessions_available.append(row)
    print(f'\nSet SESSION_INDEX below to download and load a session.')
else:
    print('No INDEX_URL set — will use synthetic data.')
    print('See instructions above to set up a Google Sheet index.')


In [ ]:
# --- Pick a session from the index ---
#   None (default): load the most recent session (last row in the sheet)
#   0, 1, 2...     : load a specific session by index
#   -1              : use synthetic demo data (no real session)
SESSION_INDEX = None

SESSION_DIR = None

# Resolve SESSION_INDEX: None → last row in the sheet
_effective_index = SESSION_INDEX
if _effective_index is None and sessions_available:
    _effective_index = len(sessions_available) - 1

if _effective_index is not None and _effective_index >= 0 and sessions_available:
    row = sessions_available[_effective_index]
    link = row.iloc[1]
    name = str(row.iloc[0])

    # Extract Google Drive file ID from various link formats
    match = re.search(r'/d/([a-zA-Z0-9_-]+)', link)
    if not match:
        match = re.search(r'id=([a-zA-Z0-9_-]+)', link)
    if match:
        file_id = match.group(1)
        zip_path = f'/content/{name}.zip'
        session_path = f'/content/{name}'

        if not os.path.exists(session_path):
            # Download with gdown
            import gdown
            gdown.download(id=file_id, output=zip_path, quiet=False)

            # Unzip
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall('/content/')
            os.remove(zip_path)
            print(f'\nUnzipped to {session_path}')

            # The zip may contain a top-level dir with a different name (route suffix)
            if not os.path.exists(session_path):
                # Find the extracted directory
                candidates = [d for d in os.listdir('/content/') if d.startswith('session_') and os.path.isdir(f'/content/{d}')]
                if candidates:
                    session_path = f'/content/{sorted(candidates)[-1]}'
                    print(f'Session dir: {session_path}')

        SESSION_DIR = session_path
        print(f'\nLoading session: {SESSION_DIR}')
    else:
        print(f'Could not extract file ID from link: {link}')


def generate_synthetic_session():
    """Generate comprehensive test session using the synthetic module."""
    from capture_analysis.synthetic import generate_and_load
    return generate_and_load('/content' if Path('/content').exists() else '/tmp')


# Load real or synthetic data
if SESSION_INDEX == -1:
    session = generate_synthetic_session()
    print('Using synthetic data (SESSION_INDEX = -1)')
elif SESSION_DIR and Path(SESSION_DIR).exists():
    session = load_session(SESSION_DIR)
    print(f"Loaded real session: {SESSION_DIR}")
else:
    session = generate_synthetic_session()
    print('No session available — using synthetic data. Set INDEX_URL and populate the sheet.')

meta = session['meta']
accel = session['accel']
lin_accel = session.get('lin_accel')
gyro = session['gyro']
mag = session['mag']
gps = session['gps']
events = session['events']

if events is not None and len(events) > 0:
    print(f"\nLabeled events: {len(events)}")
    print(events.to_string(index=False))

## Session metadata

In [ ]:
counts = meta.get('sample_counts', {})
duration_s = accel['time_s'].iloc[-1] - accel['time_s'].iloc[0] if accel is not None else 0

print(f"Session:   {meta.get('session_id', '?')}")

# Compatibility check: is this session compatible with this notebook version?
session_app_version = meta.get('app_version', 'unknown')
if session_app_version != 'unknown' and session_app_version != NOTEBOOK_VERSION:
    print(f"\u26a0\ufe0f  Session captured with app {session_app_version}, notebook is {NOTEBOOK_VERSION}")
    print(f"    (fine if the formats are compatible; update the notebook if new fields exist)")
print(f"Device:    {meta.get('device_model', '?')}")
print(f"Android:   {meta.get('android_version', '?')}")
print(f"App ver:   {meta.get('app_version', '?')}")
print(f"Start:     {meta.get('start_time_iso', '?')}")
print(f"Duration:  {duration_s:.1f} s ({duration_s/60:.1f} min)")

# Route
route = meta.get('route', {})
if route:
    origin = route.get('origin', '')
    dest = route.get('destination', '')
    abbrev = meta.get('route_abbreviation', '')
    print(f"Route:     {origin} \u2192 {dest}" + (f'  [{abbrev}]' if abbrev else ''))

# Labeling
method = meta.get('labeling_method', '')
reliability = meta.get('labeling_reliability', '')
if method:
    print(f"Labeling:  {method}" + (f' ({reliability})' if reliability else ''))

# Comment
comment = meta.get('comment', '')
if comment:
    print(f"Comment:   {comment}")

print()
print('Sample counts and actual rates:')
for sensor, count in counts.items():
    hz = count / duration_s if duration_s > 0 else 0
    print(f"  {sensor:>8s}: {count:>10,} samples  ({hz:>7.1f} Hz)")


## Accelerometer — 3-axis time series

**Raw** (with gravity, **phone frame** — X/Y/Z rotate with the device): X, Y, Z as reported by `TYPE_ACCELEROMETER`. At rest, Z reads ~9.81 m/s².

**Gravity removed** (linear acceleration): gravity estimated via low-pass filter and subtracted. At rest, all axes read ~0. The magnitude plot uses these gravity-removed values — it shows only the dynamic acceleration (bumps, potholes, braking).

In [ ]:
#@title Raw accelerometer — 3-axis + linear + magnitude
if accel is not None and len(accel) > 0:
    # Downsample for overview plot (every Nth sample)
    step = max(1, len(accel) // 10_000)
    a = accel.iloc[::step].copy()

    # Estimate gravity via low-pass filter (cutoff ~0.5 Hz)
    alpha = 0.01  # low-pass coefficient: smaller = smoother gravity estimate
    grav_x = np.zeros(len(accel), dtype=np.float32)
    grav_y = np.zeros(len(accel), dtype=np.float32)
    grav_z = np.zeros(len(accel), dtype=np.float32)
    grav_x[0] = accel['x_ms2'].iloc[0]
    grav_y[0] = accel['y_ms2'].iloc[0]
    grav_z[0] = accel['z_ms2'].iloc[0]
    for i in range(1, len(accel)):
        grav_x[i] = alpha * accel['x_ms2'].iloc[i] + (1 - alpha) * grav_x[i - 1]
        grav_y[i] = alpha * accel['y_ms2'].iloc[i] + (1 - alpha) * grav_y[i - 1]
        grav_z[i] = alpha * accel['z_ms2'].iloc[i] + (1 - alpha) * grav_z[i - 1]

    # Compute linear acceleration (gravity removed) on full data, then downsample
    accel_lin_x = accel['x_ms2'].values - grav_x
    accel_lin_y = accel['y_ms2'].values - grav_y
    accel_lin_z = accel['z_ms2'].values - grav_z

    a['lin_x'] = accel_lin_x[::step]
    a['lin_y'] = accel_lin_y[::step]
    a['lin_z'] = accel_lin_z[::step]

    fig, axes = plt.subplots(7, 1, figsize=(14, 20), sharex=True)

    # --- Raw (with gravity) ---
    axes[0].plot(a['time_s'], a['x_ms2'], linewidth=0.5, color='tab:red')
    axes[0].set_ylabel('X (m/s\u00b2)')
    axes[0].set_title('Raw accelerometer \u2014 lateral (X)')
    axes[0].grid(True, alpha=0.3)
    plot_event_markers(axes[0], events)

    axes[1].plot(a['time_s'], a['y_ms2'], linewidth=0.5, color='tab:green')
    axes[1].set_ylabel('Y (m/s\u00b2)')
    axes[1].set_title('Raw accelerometer \u2014 forward (Y)')
    axes[1].grid(True, alpha=0.3)
    plot_event_markers(axes[1], events)

    axes[2].plot(a['time_s'], a['z_ms2'], linewidth=0.5, color='tab:blue')
    axes[2].set_ylabel('Z (m/s\u00b2)')
    axes[2].set_title('Raw accelerometer \u2014 vertical (Z)')
    axes[2].grid(True, alpha=0.3)
    plot_event_markers(axes[2], events)

    # --- Gravity removed (linear acceleration) ---
    axes[3].plot(a['time_s'], a['lin_x'], linewidth=0.5, color='tab:red', alpha=0.8)
    axes[3].set_ylabel('X (m/s\u00b2)')
    axes[3].set_title('Linear acceleration \u2014 lateral (X), gravity removed')
    axes[3].grid(True, alpha=0.3)
    plot_event_markers(axes[3], events)

    axes[4].plot(a['time_s'], a['lin_y'], linewidth=0.5, color='tab:green', alpha=0.8)
    axes[4].set_ylabel('Y (m/s\u00b2)')
    axes[4].set_title('Linear acceleration \u2014 forward (Y), gravity removed')
    axes[4].grid(True, alpha=0.3)
    plot_event_markers(axes[4], events)

    axes[5].plot(a['time_s'], a['lin_z'], linewidth=0.5, color='tab:blue', alpha=0.8)
    axes[5].set_ylabel('Z (m/s\u00b2)')
    axes[5].set_title('Linear acceleration \u2014 vertical (Z), gravity removed')
    axes[5].grid(True, alpha=0.3)
    plot_event_markers(axes[5], events)

    # --- Magnitude of linear acceleration ---
    lin_mag = np.sqrt(a['lin_x']**2 + a['lin_y']**2 + a['lin_z']**2)
    axes[6].plot(a['time_s'], lin_mag, linewidth=0.5, color='tab:purple')
    axes[6].set_ylabel('|a| (m/s\u00b2)')
    axes[6].set_xlabel('Time (s)')
    axes[6].set_title('Linear acceleration magnitude (gravity removed)')
    axes[6].grid(True, alpha=0.3)
    plot_event_markers(axes[6], events)

    # Add legend for event markers
    if events is not None and len(events) > 0:
        from matplotlib.lines import Line2D
        legend_elements = [Line2D([0], [0], color='red', linestyle='--', label='pothole event'),
                           Line2D([0], [0], color='orange', linestyle='--', label='crack event')]
        axes[0].legend(handles=legend_elements, loc='upper right', fontsize=8)

    plt.tight_layout()
    show_time_on_all(axes)
    plt.show()

    # Store linear accel globally for use in GPS map and drill-down
    accel['lin_x'] = accel_lin_x
    accel['lin_y'] = accel_lin_y
    accel['lin_z'] = accel_lin_z
    accel['lin_mag'] = np.sqrt(accel_lin_x**2 + accel_lin_y**2 + accel_lin_z**2)
else:
    print('No accelerometer data.')

## Linear Acceleration — gravity removed (TYPE_LINEAR_ACCELERATION)

Android's sensor fusion subtracts gravity. **Phone frame** — values are
in the phone's coordinate system (X/Y/Z rotate with the device). Gravity
is removed but the axes are still phone-relative, not Earth-relative.

For Earth-frame decomposition (vertical/horizontal/forward/lateral),
see the cells below. from the raw accelerometer, giving
the actual motion of the device. At rest, all values are close to 0.
A pothole impact shows up as a spike on the axis pointing opposite to gravity
(typically Z if the phone is flat, Y if vertical).

In [ ]:
#@title Linear acceleration (TYPE_LINEAR_ACCELERATION)
if lin_accel is not None and len(lin_accel) > 0:
    step = max(1, len(lin_accel) // 10000)
    t = lin_accel['time_s'].iloc[::step]

    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
    axes[0].plot(t, lin_accel['x_ms2'].iloc[::step], label='X', linewidth=0.5)
    axes[0].set_ylabel('X (m/s\u00b2)')
    axes[0].axhline(0, color='gray', linewidth=0.3)
    axes[0].set_title('Linear Acceleration (gravity removed)')
    plot_event_markers(axes[0], events)

    axes[1].plot(t, lin_accel['y_ms2'].iloc[::step], color='orange', linewidth=0.5)
    axes[1].set_ylabel('Y (m/s\u00b2)')
    axes[1].axhline(0, color='gray', linewidth=0.3)
    plot_event_markers(axes[1], events)

    axes[2].plot(t, lin_accel['z_ms2'].iloc[::step], color='green', linewidth=0.5)
    axes[2].set_ylabel('Z (m/s\u00b2)')
    axes[2].axhline(0, color='gray', linewidth=0.3)
    plot_event_markers(axes[2], events)

    lin_mag = np.sqrt(lin_accel['x_ms2']**2 + lin_accel['y_ms2']**2 + lin_accel['z_ms2']**2)
    axes[3].plot(t, lin_mag.iloc[::step], color='purple', linewidth=0.5)
    axes[3].set_ylabel('|lin_a| (m/s\u00b2)')
    axes[3].set_xlabel('Time (s)')
    plot_event_markers(axes[3], events)

    plt.tight_layout()
    show_time_on_all(axes)
    plt.show()

    print(f"Baseline (first 2 s): mean={lin_mag.iloc[:int(2*500)].mean():.3f} m/s\u00b2")
    print(f"Peak magnitude: {lin_mag.max():.2f} m/s\u00b2")
else:
    print('No lin_accel data (session was recorded with app version < v018)')

## Vertical / Horizontal decomposition (orientation-tracking)

Low-pass filter the raw accelerometer to isolate gravity at each instant,
then decompose the motion into vertical (along gravity) and horizontal
(perpendicular to gravity) components.

**Three cutoff frequencies** are compared (0.2, 0.5, 1.0 Hz).

The third subplot shows the **GPS heading** (compass direction of travel):
0° = North, 90° = East, ±180° = South, -90° = West.

All values here are in the **Earth frame** (orientation-invariant).

In [ ]:
#@title Vertical / Horizontal decomposition (0.2, 0.5, 1.0 Hz)
from scipy.signal import butter, filtfilt

if accel is not None and len(accel) > 100:
    xyz = accel[['x_ms2', 'y_ms2', 'z_ms2']].values.astype(float)

    duration = accel['time_s'].iloc[-1] - accel['time_s'].iloc[0]
    fs = len(accel) / duration if duration > 0 else 500.0
    print(f'Accel sample rate: {fs:.1f} Hz')

    cutoffs_hz = [0.2, 0.5, 1.0]
    results = {}

    for cutoff in cutoffs_hz:
        nyq = fs / 2
        b, a = butter(2, cutoff / nyq, 'low')
        gx = filtfilt(b, a, xyz[:, 0])
        gy = filtfilt(b, a, xyz[:, 1])
        gz = filtfilt(b, a, xyz[:, 2])
        gravity_vec = np.column_stack([gx, gy, gz])
        gravity_mag = np.linalg.norm(gravity_vec, axis=1)
        g_hat = gravity_vec / gravity_mag[:, None]

        lin = xyz - gravity_vec
        vertical = (lin * g_hat).sum(axis=1)
        horizontal_vec = lin - vertical[:, None] * g_hat
        horizontal_mag = np.linalg.norm(horizontal_vec, axis=1)
        results[cutoff] = (vertical, horizontal_mag, g_hat, gravity_mag)

    # Save 0.5 Hz as canonical
    v_canon, h_canon, g_hat_canon, _ = results[0.5]
    accel['vertical'] = v_canon
    accel['horizontal'] = h_canon
    accel['lin_mag'] = np.sqrt(v_canon**2 + h_canon**2)

    # Phone rotation diagnostic (text only)
    g_start = g_hat_canon[0]
    g_end = g_hat_canon[-1]
    dot = np.clip(np.dot(g_start, g_end), -1, 1)
    rotation_deg = np.degrees(np.arccos(dot))
    print(f'Phone orientation change (start \u2192 end): {rotation_deg:.1f}\u00b0')

    # Prepare GPS heading for the third subplot
    has_gps_heading = (gps is not None and len(gps) > 1 and 'bearing_deg' in gps.columns)

    step = max(1, len(accel) // 10000)
    t = accel['time_s'].iloc[::step]

    for cutoff in cutoffs_hz:
        vertical, horizontal_mag, g_hat, gravity_mag = results[cutoff]

        fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
        fig.suptitle(
            f'Decomposition with cutoff = {cutoff} Hz   '
            f'(mean |g|={gravity_mag.mean():.2f} m/s\u00b2)',
            fontsize=12, fontweight='bold',
        )

        axes[0].plot(t, vertical[::step], color='steelblue', linewidth=0.5)
        axes[0].axhline(0, color='gray', linewidth=0.3)
        axes[0].set_ylabel('Vertical (m/s\u00b2)')
        plot_event_markers(axes[0], events)

        axes[1].plot(t, horizontal_mag[::step], color='crimson', linewidth=0.5)
        axes[1].axhline(0, color='gray', linewidth=0.3)
        axes[1].set_ylabel('|horizontal|\n(m/s\u00b2)')
        plot_event_markers(axes[1], events)

        # GPS heading (compass direction)
        if has_gps_heading:
            # Convert GPS bearing (0-360) to signed heading (-180 to +180)
            bearing = gps['bearing_deg'].values.copy()
            bearing = np.where(bearing > 180, bearing - 360, bearing)
            axes[2].plot(gps['time_s'], bearing, color='teal', linewidth=1.2, marker='.', markersize=2)
            axes[2].set_ylabel('GPS heading\n(\u00b0 from N)')
            axes[2].set_ylim(-190, 190)
            axes[2].set_yticks([-180, -90, 0, 90, 180])
            axes[2].set_yticklabels(['S (-180)', 'W (-90)', 'N (0)', 'E (90)', 'S (180)'])
            axes[2].axhline(0, color='gray', linewidth=0.3, linestyle=':')
            plot_event_markers(axes[2], events)
        else:
            axes[2].text(0.5, 0.5, 'No GPS heading data', ha='center', va='center',
                         transform=axes[2].transAxes, fontsize=14, color='gray')
            axes[2].set_ylabel('GPS heading')

        axes[2].set_xlabel('Time (s)')
        show_time_on_all(axes)
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

    print('\nCanonical values saved as accel["vertical"], accel["horizontal"], accel["lin_mag"] (cutoff = 0.5 Hz)')

## Jerk — derivative of linear acceleration and gyroscope

The **jerk** (derivative of acceleration) highlights *sudden changes* —
the onset and end of impacts. A pothole produces a sharp jerk spike at
entry and exit, while a gentle dip barely registers. Similarly, the
derivative of gyroscope angular velocity highlights sudden rotational
changes (phone tilts from the impact).

Jerk can be more distinctive than raw acceleration for detection because
normal road vibrations have smaller jerk than pothole impacts.

In [ ]:
#@title Jerk — derivative of linear acceleration and gyroscope
if accel is not None and len(accel) > 100:
    dt = np.median(np.diff(accel['time_s'].values[:10000]))
    step = max(1, len(accel) // 10000)
    t = accel['time_s'].iloc[::step]

    # Linear acceleration jerk (if decomposition ran, use those; else use raw - gravity approximation)
    if 'vertical' in accel.columns:
        vert = accel['vertical'].values
        horiz = accel['horizontal'].values
    else:
        lin_mag_vals = np.sqrt(accel['x_ms2']**2 + accel['y_ms2']**2 + accel['z_ms2']**2) - 9.81
        vert = lin_mag_vals.values
        horiz = np.zeros_like(vert)

    # Compute jerk: d(acceleration)/dt
    jerk_vert = np.gradient(vert, dt)
    jerk_horiz = np.gradient(horiz, dt)
    jerk_total = np.sqrt(jerk_vert**2 + jerk_horiz**2)

    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    fig.suptitle('Jerk — derivative of acceleration (m/s\u00b3)', fontweight='bold')

    axes[0].plot(t, jerk_vert[::step], color='steelblue', linewidth=0.5)
    axes[0].axhline(0, color='gray', linewidth=0.3)
    axes[0].set_ylabel('Vertical\njerk (m/s\u00b3)')
    plot_event_markers(axes[0], events)

    axes[1].plot(t, jerk_horiz[::step], color='crimson', linewidth=0.5)
    axes[1].axhline(0, color='gray', linewidth=0.3)
    axes[1].set_ylabel('Horizontal\njerk (m/s\u00b3)')
    plot_event_markers(axes[1], events)

    axes[2].plot(t, jerk_total[::step], color='purple', linewidth=0.5)
    axes[2].set_ylabel('|jerk|\n(m/s\u00b3)')
    axes[2].set_xlabel('Time (s)')
    plot_event_markers(axes[2], events)

    show_time_on_all(axes)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    # Also compute gyroscope jerk
    if gyro is not None and len(gyro) > 100:
        dt_g = np.median(np.diff(gyro['time_s'].values[:10000]))
        step_g = max(1, len(gyro) // 10000)
        t_g = gyro['time_s'].iloc[::step_g]

        jerk_gx = np.gradient(gyro['x_rads'].values, dt_g)
        jerk_gy = np.gradient(gyro['y_rads'].values, dt_g)
        jerk_gz = np.gradient(gyro['z_rads'].values, dt_g)
        jerk_g_mag = np.sqrt(jerk_gx**2 + jerk_gy**2 + jerk_gz**2)

        fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
        fig.suptitle('Gyroscope jerk — derivative of angular velocity (rad/s\u00b2)', fontweight='bold')

        axes[0].plot(t_g, jerk_gx[::step_g], label='X', linewidth=0.5)
        axes[0].plot(t_g, jerk_gy[::step_g], label='Y', linewidth=0.5)
        axes[0].plot(t_g, jerk_gz[::step_g], label='Z', linewidth=0.5)
        axes[0].set_ylabel('Gyro jerk\n(rad/s\u00b2)')
        axes[0].legend(fontsize=9, ncol=3)
        axes[0].axhline(0, color='gray', linewidth=0.3)
        plot_event_markers(axes[0], events)

        axes[1].plot(t_g, jerk_g_mag[::step_g], color='gray', linewidth=0.5)
        axes[1].set_ylabel('|gyro jerk|\n(rad/s\u00b2)')
        axes[1].set_xlabel('Time (s)')
        plot_event_markers(axes[1], events)

        show_time_on_all(axes)
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

    # Save jerk for the combined activity cell
    accel['jerk_total'] = jerk_total

## Combined activity — rolling 1-second energy

Instead of looking at each parameter separately, we **combine all motion
signals** into a single "activity" metric per 1-second window:

- **Vertical accel RMS** (1s window)
- **Horizontal accel RMS** (1s window)
- **Jerk RMS** (1s window)
- **Gyro angular velocity RMS** (1s window)

Each is normalized (0 = quiet, 1 = peak), then summed → a single
"combined activity" score. Potholes should produce a clear spike in
this combined signal that rises above normal road noise.

In [ ]:
#@title Combined 1-second activity score
if accel is not None and len(accel) > 500:
    duration = accel['time_s'].iloc[-1] - accel['time_s'].iloc[0]
    fs_accel = int(len(accel) / duration) if duration > 0 else 500
    window = fs_accel  # 1-second window

    # RMS over 1-second rolling windows
    def rolling_rms(series, w):
        return np.sqrt(pd.Series(series**2).rolling(w, center=True, min_periods=w//2).mean())

    # Vertical acceleration RMS
    if 'vertical' in accel.columns:
        vert_rms = rolling_rms(accel['vertical'].values, window)
    else:
        mag = np.sqrt(accel['x_ms2']**2 + accel['y_ms2']**2 + accel['z_ms2']**2) - 9.81
        vert_rms = rolling_rms(mag.values, window)

    # Horizontal acceleration RMS
    if 'horizontal' in accel.columns:
        horiz_rms = rolling_rms(accel['horizontal'].values, window)
    else:
        horiz_rms = np.zeros_like(vert_rms)

    # Jerk RMS
    if 'jerk_total' in accel.columns:
        jerk_rms = rolling_rms(accel['jerk_total'].values, window)
    else:
        dt = np.median(np.diff(accel['time_s'].values[:10000]))
        mag = np.sqrt(accel['x_ms2']**2 + accel['y_ms2']**2 + accel['z_ms2']**2)
        jerk_rms = rolling_rms(np.gradient(mag.values, dt), window)

    # Gyro angular velocity RMS
    if gyro is not None and len(gyro) > 100:
        fs_gyro = int(len(gyro) / (gyro['time_s'].iloc[-1] - gyro['time_s'].iloc[0]))
        w_gyro = fs_gyro
        gyro_mag = np.sqrt(gyro['x_rads']**2 + gyro['y_rads']**2 + gyro['z_rads']**2)
        gyro_rms = rolling_rms(gyro_mag.values, w_gyro)
        # Resample gyro to accel timeline
        gyro_rms_resampled = np.interp(accel['time_s'].values, gyro['time_s'].values, gyro_rms.values)
    else:
        gyro_rms_resampled = np.zeros_like(vert_rms)

    # Normalize each to [0, 1] using 99th percentile as max
    def norm99(x):
        p99 = np.nanpercentile(x, 99)
        return np.clip(x / p99, 0, 1) if p99 > 0 else x * 0

    vert_norm = norm99(vert_rms)
    horiz_norm = norm99(horiz_rms)
    jerk_norm = norm99(jerk_rms)
    gyro_norm = norm99(gyro_rms_resampled)

    # Combined score: sum of normalized components
    combined = vert_norm + horiz_norm + jerk_norm + gyro_norm

    step = max(1, len(accel) // 10000)
    t = accel['time_s'].iloc[::step]

    fig, axes = plt.subplots(5, 1, figsize=(14, 12), sharex=True)
    fig.suptitle('Combined activity — rolling 1-second RMS (normalized)', fontweight='bold')

    axes[0].plot(t, vert_norm[::step], color='steelblue', linewidth=0.8)
    axes[0].set_ylabel('Vertical\naccel')
    axes[0].set_ylim(0, 1.1)
    plot_event_markers(axes[0], events)

    axes[1].plot(t, horiz_norm[::step], color='crimson', linewidth=0.8)
    axes[1].set_ylabel('Horizontal\naccel')
    axes[1].set_ylim(0, 1.1)
    plot_event_markers(axes[1], events)

    axes[2].plot(t, jerk_norm[::step], color='purple', linewidth=0.8)
    axes[2].set_ylabel('Jerk')
    axes[2].set_ylim(0, 1.1)
    plot_event_markers(axes[2], events)

    axes[3].plot(t, gyro_norm[::step], color='olive', linewidth=0.8)
    axes[3].set_ylabel('Gyroscope')
    axes[3].set_ylim(0, 1.1)
    plot_event_markers(axes[3], events)

    axes[4].fill_between(t, 0, combined[::step], color='black', alpha=0.5)
    axes[4].plot(t, combined[::step], color='black', linewidth=0.8)
    axes[4].set_ylabel('COMBINED\nactivity')
    axes[4].set_xlabel('Time (s)')
    plot_event_markers(axes[4], events)

    show_time_on_all(axes)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    # Summary statistics
    print(f'\nCombined activity stats:')
    print(f'  Median: {np.nanmedian(combined):.2f}')
    print(f'  95th percentile: {np.nanpercentile(combined, 95):.2f}')
    print(f'  Max: {np.nanmax(combined):.2f}')

    # Save for potential use in GPS map coloring
    accel['combined_activity'] = combined

## Gyroscope — 3-axis time series

In [ ]:
#@title Gyroscope — 3-axis
if gyro is not None and len(gyro) > 0:
    step = max(1, len(gyro) // 10_000)
    g = gyro.iloc[::step]

    fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)

    axes[0].plot(g['time_s'], g['x_rads'], linewidth=0.5, color='tab:red')
    axes[0].set_ylabel('X (rad/s)')
    axes[0].set_title('Gyroscope (roll)')
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(g['time_s'], g['y_rads'], linewidth=0.5, color='tab:green')
    axes[1].set_ylabel('Y (rad/s)')
    plot_event_markers(axes[1], events)
    axes[1].set_title('Gyroscope (pitch)')
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(g['time_s'], g['z_rads'], linewidth=0.5, color='tab:blue')
    axes[2].set_ylabel('Z (rad/s)')
    plot_event_markers(axes[2], events)
    axes[2].set_xlabel('Time (s)')
    axes[2].set_title('Gyroscope (yaw)')
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    show_time_on_all(axes)
    plt.show()
else:
    print('No gyroscope data.')

## Speed profile (GPS)

In [ ]:
#@title GPS speed profile
if gps is not None and len(gps) > 0 and 'speed_mps' in gps.columns:
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(gps['time_s'], gps['speed_mps'] * 3.6, linewidth=1, color='tab:orange')
    ax.set_ylabel('Speed (km/h)')
    ax.set_xlabel('Time (s)')
    ax.set_title('Vehicle speed')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(bottom=0)
    plot_event_markers(ax, events)
    plt.tight_layout()
    plt.show()
else:
    print('No GPS speed data.')

## GPS-derived accelerations (Earth frame)

Computed purely from GPS at ~1 Hz:
- **Longitudinal** = d(speed)/dt — braking (<0) and acceleration (>0)
- **Lateral** = speed × d(heading)/dt — centripetal force in turns

These are in the **Earth frame** (independent of phone orientation).
Low resolution (1 Hz) but drift-free — useful as a reference to compare
against the MEMS accelerometers.

In [ ]:
#@title GPS-derived accelerations (Earth frame, 1 Hz)
if gps is not None and len(gps) > 2 and 'speed_mps' in gps.columns and 'bearing_deg' in gps.columns:
    gps_t = gps['time_s'].values
    gps_speed = gps['speed_mps'].values
    gps_bearing = gps['bearing_deg'].values.copy()

    # Longitudinal acceleration: d(speed)/dt
    gps_dt = np.gradient(gps_t)
    gps_dt[gps_dt == 0] = 1.0  # avoid division by zero
    gps_a_long = np.gradient(gps_speed, gps_t)

    # Lateral acceleration: speed * d(heading)/dt
    # Handle bearing wraparound (e.g., 359° → 1° = +2°, not -358°)
    bearing_rad = np.deg2rad(gps_bearing)
    dbearing = np.angle(np.exp(1j * np.diff(bearing_rad)))  # shortest angular diff
    dbearing = np.concatenate([[0], dbearing])  # prepend 0 for same length
    bearing_rate = dbearing / gps_dt  # rad/s
    gps_a_lat = gps_speed * bearing_rate

    gps_a_total = np.sqrt(gps_a_long**2 + gps_a_lat**2)

    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
    fig.suptitle('GPS-derived accelerations (Earth frame, 1 Hz)', fontweight='bold')

    axes[0].plot(gps_t, gps_a_long, color='tab:blue', linewidth=1.2)
    axes[0].axhline(0, color='gray', linewidth=0.3)
    axes[0].set_ylabel('Longitudinal\n(m/s\u00b2)\n\u2190 braking | accel \u2192')
    plot_event_markers(axes[0], events)

    axes[1].plot(gps_t, gps_a_lat, color='tab:orange', linewidth=1.2)
    axes[1].axhline(0, color='gray', linewidth=0.3)
    axes[1].set_ylabel('Lateral\n(m/s\u00b2)\n\u2190 left | right \u2192')
    plot_event_markers(axes[1], events)

    axes[2].plot(gps_t, gps_a_total, color='tab:green', linewidth=1.2)
    axes[2].set_ylabel('|total horiz|\n(m/s\u00b2)')
    axes[2].set_xlabel('Time (s)')
    plot_event_markers(axes[2], events)

    show_time_on_all(axes)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

    # Save for comparison cell
    gps['a_longitudinal'] = gps_a_long
    gps['a_lateral'] = gps_a_lat
    gps['a_total_horiz'] = gps_a_total
else:
    print('No GPS speed/bearing data available for acceleration computation.')

## MEMS vs GPS comparison (Earth frame)

The MEMS accelerometer sees **everything**: road surface, potholes, engine
vibrations, AND vehicle dynamics (braking, turning). GPS only sees vehicle
dynamics (at 1 Hz). The **difference** (MEMS - GPS) isolates the road
surface signal.

To compare MEMS and GPS in the same Earth frame, we need to transform MEMS
data from the phone frame to the Earth frame using:
- **Gravity** (from low-pass accel) → defines "up"
- **Magnetometer** → defines "north" in the phone frame
- **GPS heading** → defines "forward" direction of travel

Legend convention:
- **(Earth frame)** = projected into world coordinates (north/east/up)
- **(phone frame)** = raw device X/Y/Z (includes gravity tilt effects)

In [ ]:
#@title MEMS vs GPS — forward/lateral comparison (Earth frame)
from scipy.signal import butter, filtfilt

_can_compare = (
    accel is not None and len(accel) > 100
    and gps is not None and len(gps) > 2
    and 'a_longitudinal' in gps.columns
)

if _can_compare:
    xyz = accel[['x_ms2', 'y_ms2', 'z_ms2']].values.astype(float)
    duration = accel['time_s'].iloc[-1] - accel['time_s'].iloc[0]
    fs = len(accel) / duration if duration > 0 else 500.0

    # 1. Gravity estimation (low-pass 0.5 Hz)
    nyq = fs / 2
    b, a = butter(2, 0.5 / nyq, 'low')
    grav = np.column_stack([filtfilt(b, a, xyz[:, i]) for i in range(3)])
    grav_mag = np.linalg.norm(grav, axis=1, keepdims=True)
    g_hat = grav / grav_mag

    # Linear acceleration (gravity removed)
    lin = xyz - grav

    # 2. Use magnetometer to define North in phone frame
    _has_mag = mag is not None and len(mag) > 100
    if _has_mag:
        # Resample mag to accel timestamps
        mx = np.interp(accel['time_s'].values, mag['time_s'].values, mag['x_ut'].values)
        my = np.interp(accel['time_s'].values, mag['time_s'].values, mag['y_ut'].values)
        mz = np.interp(accel['time_s'].values, mag['time_s'].values, mag['z_ut'].values)
        mag_vec = np.column_stack([mx, my, mz])

        # East = normalize(cross(mag, gravity))
        east = np.cross(mag_vec, grav)
        east_mag = np.linalg.norm(east, axis=1, keepdims=True)
        east_mag[east_mag < 1e-6] = 1  # avoid div by zero
        east = east / east_mag

        # North = normalize(cross(gravity, east))
        north = np.cross(grav, east)
        north_mag = np.linalg.norm(north, axis=1, keepdims=True)
        north_mag[north_mag < 1e-6] = 1
        north = north / north_mag

        # Project linear accel into world frame (magnetic north/east)
        lin_north = (lin * north).sum(axis=1)
        lin_east = (lin * east).sum(axis=1)

        # 3. Rotate from north/east to forward/lateral using GPS heading
        gps_bearing_rad = np.interp(
            accel['time_s'].values, gps['time_s'].values,
            np.deg2rad(gps['bearing_deg'].values)
        )
        cos_h = np.cos(gps_bearing_rad)
        sin_h = np.sin(gps_bearing_rad)

        # Forward = N*cos(heading) + E*sin(heading)
        # Lateral = -N*sin(heading) + E*cos(heading)  (positive = right)
        mems_forward = lin_north * cos_h + lin_east * sin_h
        mems_lateral = -lin_north * sin_h + lin_east * cos_h

        # 4. Resample GPS accelerations to MEMS timeline
        gps_long_resampled = np.interp(accel['time_s'].values, gps['time_s'].values, gps['a_longitudinal'].values)
        gps_lat_resampled = np.interp(accel['time_s'].values, gps['time_s'].values, gps['a_lateral'].values)

        # 5. Differences (MEMS - GPS = road surface signal)
        diff_forward = mems_forward - gps_long_resampled
        diff_lateral = mems_lateral - gps_lat_resampled

        step = max(1, len(accel) // 10000)
        t = accel['time_s'].iloc[::step]

        # --- Plot 1: Forward (longitudinal) comparison ---
        fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
        fig.suptitle('Forward (longitudinal) — MEMS vs GPS (Earth frame)', fontweight='bold')

        axes[0].plot(t, mems_forward[::step], color='steelblue', linewidth=0.5, label='MEMS forward (Earth frame)')
        axes[0].set_ylabel('MEMS forward\n(m/s\u00b2, Earth)')
        axes[0].axhline(0, color='gray', linewidth=0.3)
        axes[0].legend(fontsize=9)
        plot_event_markers(axes[0], events)

        axes[1].plot(gps['time_s'], gps['a_longitudinal'], color='tab:green', linewidth=1.2, label='GPS longitudinal (Earth frame, 1 Hz)')
        axes[1].set_ylabel('GPS long.\n(m/s\u00b2, Earth)')
        axes[1].axhline(0, color='gray', linewidth=0.3)
        axes[1].legend(fontsize=9)
        plot_event_markers(axes[1], events)

        axes[2].plot(t, diff_forward[::step], color='purple', linewidth=0.5, label='Difference: MEMS - GPS (road surface)')
        axes[2].set_ylabel('\u0394 forward\n(m/s\u00b2)')
        axes[2].set_xlabel('Time (s)')
        axes[2].axhline(0, color='gray', linewidth=0.3)
        axes[2].legend(fontsize=9)
        plot_event_markers(axes[2], events)

        show_time_on_all(axes)
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

        # --- Plot 2: Lateral comparison ---
        fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
        fig.suptitle('Lateral — MEMS vs GPS (Earth frame)', fontweight='bold')

        axes[0].plot(t, mems_lateral[::step], color='crimson', linewidth=0.5, label='MEMS lateral (Earth frame)')
        axes[0].set_ylabel('MEMS lateral\n(m/s\u00b2, Earth)')
        axes[0].axhline(0, color='gray', linewidth=0.3)
        axes[0].legend(fontsize=9)
        plot_event_markers(axes[0], events)

        axes[1].plot(gps['time_s'], gps['a_lateral'], color='tab:orange', linewidth=1.2, label='GPS lateral (Earth frame, 1 Hz)')
        axes[1].set_ylabel('GPS lateral\n(m/s\u00b2, Earth)')
        axes[1].axhline(0, color='gray', linewidth=0.3)
        axes[1].legend(fontsize=9)
        plot_event_markers(axes[1], events)

        axes[2].plot(t, diff_lateral[::step], color='olive', linewidth=0.5, label='Difference: MEMS - GPS (road surface)')
        axes[2].set_ylabel('\u0394 lateral\n(m/s\u00b2)')
        axes[2].set_xlabel('Time (s)')
        axes[2].axhline(0, color='gray', linewidth=0.3)
        axes[2].legend(fontsize=9)
        plot_event_markers(axes[2], events)

        show_time_on_all(axes)
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

        # Save for later use
        accel['mems_forward'] = mems_forward
        accel['mems_lateral'] = mems_lateral
        accel['diff_forward'] = diff_forward
        accel['diff_lateral'] = diff_lateral

        print('\nRMS comparison:')
        print(f'  Forward — MEMS: {np.sqrt(np.mean(mems_forward**2)):.3f}  GPS: {np.sqrt(np.mean(gps_long_resampled**2)):.3f}  Diff: {np.sqrt(np.mean(diff_forward**2)):.3f} m/s\u00b2')
        print(f'  Lateral — MEMS: {np.sqrt(np.mean(mems_lateral**2)):.3f}  GPS: {np.sqrt(np.mean(gps_lat_resampled**2)):.3f}  Diff: {np.sqrt(np.mean(diff_lateral**2)):.3f} m/s\u00b2')
    else:
        print('Magnetometer data not available — cannot transform MEMS to Earth frame.')
        print('MEMS forward/lateral decomposition requires mag data to define north.')
        print('\nShowing GPS-derived accelerations only (see cell above).')
else:
    print('GPS speed/bearing data or accel data not available for comparison.')

In [ ]:
#@title Audio waveform + playback
import wave
from IPython.display import Audio, display as ipy_display

# Try to load audio file from session
audio_data = None
audio_sr = 16000
audio_time_s = None

if SESSION_DIR and Path(SESSION_DIR).exists():
    audio_files = list(Path(SESSION_DIR).glob("audio_*.wav"))
    if audio_files:
        with wave.open(str(audio_files[0]), 'rb') as wf:
            audio_sr = wf.getframerate()
            n_frames = wf.getnframes()
            raw = wf.readframes(n_frames)
            audio_data = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
            audio_time_s = np.arange(len(audio_data)) / audio_sr
        print(f"Loaded audio: {len(audio_data)} samples, {audio_sr} Hz, {len(audio_data)/audio_sr:.1f}s")
else:
    # Generate synthetic audio for demo: silence + voice-like bursts at pothole times
    duration_audio = 120.0
    n_samples = int(duration_audio * audio_sr)
    audio_data = np.random.normal(0, 0.01, n_samples).astype(np.float32)  # background noise

    # Simulate voice commands ~1.5s after each pothole (slightly after BT button press)
    for t_event in [30.0, 65.0, 95.0]:
        t_voice = t_event + 1.5
        idx = int(t_voice * audio_sr)
        duration_voice = int(0.5 * audio_sr)  # 500ms "nid!" utterance
        t_local = np.arange(duration_voice) / audio_sr
        voice = 0.3 * np.sin(2 * np.pi * 200 * t_local) * np.exp(-3 * t_local)  # crude voice sim
        voice += 0.1 * np.sin(2 * np.pi * 400 * t_local) * np.exp(-4 * t_local)
        end = min(idx + len(voice), n_samples)
        audio_data[idx:end] += voice[:end - idx].astype(np.float32)

    # Simulate impact sounds at exact pothole times
    for t_event in [30.0, 65.0, 95.0]:
        idx = int(t_event * audio_sr)
        duration_impact = int(0.05 * audio_sr)  # 50ms impact
        impact = 0.15 * np.random.normal(0, 1, duration_impact)  # broadband thunk
        end = min(idx + len(impact), n_samples)
        audio_data[idx:end] += impact[:end - idx].astype(np.float32)

    audio_time_s = np.arange(len(audio_data)) / audio_sr
    print(f"Synthetic audio: {len(audio_data)} samples, {audio_sr} Hz, {len(audio_data)/audio_sr:.1f}s")

if audio_data is not None:
    # Downsample for plotting (plot envelope, not every sample)
    chunk = max(1, len(audio_data) // 5000)
    envelope = np.array([
        np.max(np.abs(audio_data[i:i + chunk]))
        for i in range(0, len(audio_data) - chunk, chunk)
    ])
    t_env = np.arange(len(envelope)) * chunk / audio_sr

    fig, ax = plt.subplots(figsize=(14, 3))
    ax.fill_between(t_env, -envelope, envelope, color='tab:cyan', alpha=0.6)
    ax.set_ylabel('Amplitude')
    ax.set_xlabel('Time (s)')
    ax.set_title('Audio waveform (envelope)')
    ax.set_ylim(-0.5, 0.5)
    ax.grid(True, alpha=0.3)
    plot_event_markers(ax, events)
    plt.tight_layout()
    plt.show()

    # Audio playback widget (works in Jupyter and Colab)
    print("Audio playback:")
    ipy_display(Audio(audio_data, rate=audio_sr))

## Audio waveform

Continuous microphone recording captures voice labels ("nid!", "gros nid!") and road impact sounds.
The waveform shows amplitude over time — voice commands appear as clear bursts above road noise.

## Spectrogram — Vertical acceleration (Z-axis)

Potholes produce broadband mid-frequency bursts (5-15 Hz).

In [ ]:
#@title Spectrogram — Z-axis
if accel is not None and len(accel) > 500:
    # Estimate actual sample rate from timestamps
    dt_ns = np.diff(accel['timestamp_ns'].values[:10000])
    fs = 1e9 / np.median(dt_ns)
    print(f'Estimated sample rate: {fs:.0f} Hz')

    # Compute spectrogram
    nperseg = min(512, len(accel) // 4)
    f, t, Sxx = signal.spectrogram(
        accel['z_ms2'].values, fs=fs, nperseg=nperseg,
        noverlap=nperseg // 2, scaling='density',
    )

    # Limit frequency range to 0-50 Hz (most interesting for potholes)
    f_max = 50
    f_mask = f <= f_max

    fig, ax = plt.subplots(figsize=(14, 5))
    im = ax.pcolormesh(
        t, f[f_mask], 10 * np.log10(Sxx[f_mask] + 1e-10),
        shading='gouraud', cmap='inferno',
    )
    ax.set_ylabel('Frequency (Hz)')
    ax.set_xlabel('Time (s)')
    ax.set_title('Z-axis acceleration spectrogram')
    plt.colorbar(im, ax=ax, label='Power (dB)')

    # Mark pothole frequency band
    ax.axhline(5, color='white', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.axhline(15, color='white', linestyle='--', alpha=0.5, linewidth=0.8)
    ax.text(t[0] + 0.5, 10, 'pothole band', color='white', fontsize=9, alpha=0.7)

    plot_event_markers(ax, events)
    plt.tight_layout()
    plt.show()
else:
    print('Not enough accelerometer data for spectrogram.')

## GPS track map

Track colored by acceleration magnitude. Red = high acceleration spikes, green = calm driving.
Click the markers to see peak acceleration values.

In [ ]:
#@title GPS track map
if (gps is not None and len(gps) > 1 and accel is not None and len(accel) > 0
        and 'lat_deg' in gps.columns):

    # Use linear acceleration magnitude (gravity removed) if available, else raw
    if 'lin_mag' in accel.columns:
        accel_mag = accel['lin_mag']
    else:
        accel_mag = np.sqrt(accel['x_ms2']**2 + accel['y_ms2']**2 + accel['z_ms2']**2)

    # For each GPS fix, find the max acceleration in a 1-second window around it
    gps_accel_max = []
    for _, row in gps.iterrows():
        t = row['time_s']
        mask = (accel['time_s'] >= t - 0.5) & (accel['time_s'] < t + 0.5)
        if mask.any():
            gps_accel_max.append(accel_mag[mask].max())
        else:
            gps_accel_max.append(0.0)
    gps_accel_max = np.array(gps_accel_max)

    # Normalize for color mapping
    vmin, vmax = 0.5, max(5.0, np.percentile(gps_accel_max, 98))
    norm = (gps_accel_max - vmin) / (vmax - vmin)
    norm = np.clip(norm, 0, 1)

    def value_to_color(v):
        """Green (0) to Yellow (0.5) to Red (1)."""
        r = int(min(255, v * 2 * 255))
        g = int(min(255, (1 - v) * 2 * 255))
        return f'#{r:02x}{g:02x}00'

    # Create map centered on the track
    center_lat = gps['lat_deg'].mean()
    center_lon = gps['lon_deg'].mean()
    m = folium.Map(location=[center_lat, center_lon], zoom_start=15, tiles='CartoDB voyager')

    # Draw colored track segments
    for i in range(len(gps) - 1):
        color = value_to_color(norm[i])
        folium.PolyLine(
            locations=[
                [gps.iloc[i]['lat_deg'], gps.iloc[i]['lon_deg']],
                [gps.iloc[i + 1]['lat_deg'], gps.iloc[i + 1]['lon_deg']],
            ],
            color=color, weight=5, opacity=0.8,
        ).add_to(m)

    # Add markers at top-5 acceleration peaks
    top_indices = np.argsort(gps_accel_max)[-5:]
    for idx in top_indices:
        row = gps.iloc[idx]
        val = gps_accel_max[idx]
        folium.Marker(
            location=[row['lat_deg'], row['lon_deg']],
            popup=f"Peak: {val:.1f} m/s\u00b2 @ t={row['time_s']:.1f}s",
            icon=folium.Icon(color='red', icon='exclamation-sign'),
        ).add_to(m)

    # Start marker
    folium.Marker(
        location=[gps.iloc[0]['lat_deg'], gps.iloc[0]['lon_deg']],
        popup='Start',
        icon=folium.Icon(color='green', icon='play'),
    ).add_to(m)

    display(m)
else:
    print('No GPS data available for map.')

## Interactive drill-down

Use the sliders to explore the session:
- **Center** — position in the session (drag to scroll through time)
- **Window** — how many seconds to show around the center

The drill-down shows all signals aligned on the same time axis:
raw accel, linear accel, gyroscope, speed, and combined activity.

In [ ]:
#@title Interactive drill-down — drag sliders to explore
from ipywidgets import interact, FloatRangeSlider, Layout
from matplotlib.gridspec import GridSpec

if accel is not None and len(accel) > 0:
    _t_min = float(accel['time_s'].iloc[0])
    _t_max = float(accel['time_s'].iloc[-1])
    _duration = _t_max - _t_min

    # Precompute GPS track for mini map
    _has_gps_track = (gps is not None and len(gps) > 1 and 'lat_deg' in gps.columns)

    def interactive_zoom(time_range):
        t_start, t_end = time_range
        if t_end <= t_start:
            print('Select a valid time range.')
            return

        a_mask = (accel['time_s'] >= t_start) & (accel['time_s'] <= t_end)
        a_win = accel[a_mask]

        has_forward = 'mems_forward' in accel.columns
        has_vertical = 'vertical' in accel.columns
        has_activity = 'combined_activity' in accel.columns
        has_diff = 'diff_forward' in accel.columns

        # Count signal subplots
        n_signal = 3  # raw accel, gyro, speed (always present)
        if has_vertical: n_signal += 1  # vertical
        if has_forward: n_signal += 1   # forward/lateral
        if has_activity: n_signal += 1  # combined activity
        if has_diff: n_signal += 2      # delta forward + delta lateral

        # Layout: mini map on top (short), then signal subplots
        height_ratios = [1.5] + [1] * n_signal if _has_gps_track else [1] * n_signal
        n_rows = len(height_ratios)
        fig = plt.figure(figsize=(14, 2.5 * n_rows))
        gs = GridSpec(n_rows, 1, figure=fig, height_ratios=height_ratios, hspace=0.3)

        ax_idx = 0

        # --- Mini map ---
        if _has_gps_track:
            ax_map = fig.add_subplot(gs[ax_idx])
            ax_map.set_aspect('equal')
            ax_map.plot(gps['lon_deg'], gps['lat_deg'], color='gray', linewidth=1.5, alpha=0.5)

            # Highlight selected segment
            seg_mask = (gps['time_s'] >= t_start) & (gps['time_s'] <= t_end)
            seg = gps[seg_mask]
            if len(seg) > 0:
                ax_map.plot(seg['lon_deg'], seg['lat_deg'], color='red', linewidth=3)

            # Green bars at start and end positions
            for t_mark, label in [(t_start, 'A'), (t_end, 'B')]:
                idx = (gps['time_s'] - t_mark).abs().idxmin()
                row = gps.loc[idx]
                lat, lon = row['lat_deg'], row['lon_deg']
                # Draw perpendicular bar using bearing
                if 'bearing_deg' in gps.columns:
                    brg = np.deg2rad(row['bearing_deg'] + 90)  # perpendicular
                    bar_len = 0.0002  # ~20m in degrees at Montreal latitude
                    dx = bar_len * np.cos(brg)
                    dy = bar_len * np.sin(brg)
                    ax_map.plot([lon - dx, lon + dx], [lat - dy, lat + dy],
                                color='lime', linewidth=3, solid_capstyle='round')
                ax_map.annotate(label, (lon, lat), fontsize=10, fontweight='bold',
                                color='lime', ha='center', va='bottom',
                                xytext=(0, 5), textcoords='offset points')

            # Start/end markers for full trip
            ax_map.plot(gps['lon_deg'].iloc[0], gps['lat_deg'].iloc[0], 'go', markersize=8)
            ax_map.plot(gps['lon_deg'].iloc[-1], gps['lat_deg'].iloc[-1], 'rs', markersize=8)
            ax_map.set_title(f'Trip overview — selected: {t_start:.1f}s \u2192 {t_end:.1f}s', fontsize=10)
            ax_map.tick_params(labelsize=7)
            ax_idx += 1

        signal_axes = []

        # 1. Raw accel magnitude
        ax = fig.add_subplot(gs[ax_idx], sharex=signal_axes[0] if signal_axes else None)
        signal_axes.append(ax)
        raw_mag = np.sqrt(a_win['x_ms2']**2 + a_win['y_ms2']**2 + a_win['z_ms2']**2)
        ax.plot(a_win['time_s'], raw_mag, linewidth=0.5, color='tab:blue', label='|raw accel| (phone frame, with gravity)')
        ax.set_ylabel('|raw|\n(m/s\u00b2)')
        ax.legend(fontsize=7, loc='upper right')
        plot_event_markers(ax, events, t_start, t_end)
        ax_idx += 1

        # 2. Vertical accel (Earth frame)
        if has_vertical:
            ax = fig.add_subplot(gs[ax_idx], sharex=signal_axes[0])
            signal_axes.append(ax)
            ax.plot(a_win['time_s'], a_win['vertical'], linewidth=0.5, color='steelblue', label='Vertical (Earth frame)')
            ax.axhline(0, color='gray', linewidth=0.3)
            ax.set_ylabel('Vertical\n(m/s\u00b2)')
            ax.legend(fontsize=7, loc='upper right')
            plot_event_markers(ax, events, t_start, t_end)
            ax_idx += 1

        # 3. Forward / Lateral (Earth frame)
        if has_forward:
            ax = fig.add_subplot(gs[ax_idx], sharex=signal_axes[0])
            signal_axes.append(ax)
            ax.plot(a_win['time_s'], a_win['mems_forward'], linewidth=0.5, color='tab:blue', label='Forward (Earth frame)')
            ax.plot(a_win['time_s'], a_win['mems_lateral'], linewidth=0.5, color='tab:orange', alpha=0.7, label='Lateral (Earth frame)')
            ax.axhline(0, color='gray', linewidth=0.3)
            ax.set_ylabel('Fwd / Lat\n(m/s\u00b2)')
            ax.legend(fontsize=7, loc='upper right', ncol=2)
            plot_event_markers(ax, events, t_start, t_end)
            ax_idx += 1

        # 4. Gyroscope
        ax = fig.add_subplot(gs[ax_idx], sharex=signal_axes[0])
        signal_axes.append(ax)
        if gyro is not None:
            g_mask = (gyro['time_s'] >= t_start) & (gyro['time_s'] <= t_end)
            g_win = gyro[g_mask]
            if len(g_win) > 0:
                ax.plot(g_win['time_s'], g_win['x_rads'], linewidth=0.5, label='X')
                ax.plot(g_win['time_s'], g_win['y_rads'], linewidth=0.5, label='Y')
                ax.plot(g_win['time_s'], g_win['z_rads'], linewidth=0.5, label='Z')
        ax.axhline(0, color='gray', linewidth=0.3)
        ax.set_ylabel('Gyro\n(rad/s, phone)')
        ax.legend(fontsize=7, ncol=3, loc='upper right')
        plot_event_markers(ax, events, t_start, t_end)
        ax_idx += 1

        # 5. Speed
        ax = fig.add_subplot(gs[ax_idx], sharex=signal_axes[0])
        signal_axes.append(ax)
        if gps is not None and 'speed_mps' in gps.columns:
            gp_mask = (gps['time_s'] >= t_start) & (gps['time_s'] <= t_end)
            gp_win = gps[gp_mask]
            if len(gp_win) > 0:
                ax.plot(gp_win['time_s'], gp_win['speed_mps'] * 3.6, linewidth=1.2, color='tab:orange', label='Speed (GPS, Earth frame)')
        ax.set_ylabel('Speed\n(km/h)')
        ax.legend(fontsize=7, loc='upper right')
        plot_event_markers(ax, events, t_start, t_end)
        ax_idx += 1

        # 6. Combined activity
        if has_activity:
            ax = fig.add_subplot(gs[ax_idx], sharex=signal_axes[0])
            signal_axes.append(ax)
            ax.fill_between(a_win['time_s'], 0, a_win['combined_activity'], color='black', alpha=0.3)
            ax.plot(a_win['time_s'], a_win['combined_activity'], color='black', linewidth=0.8, label='Combined activity (Earth frame)')
            ax.set_ylabel('Combined\nactivity')
            ax.legend(fontsize=7, loc='upper right')
            plot_event_markers(ax, events, t_start, t_end)
            ax_idx += 1

        # 7-8. MEMS-GPS differences
        if has_diff:
            ax = fig.add_subplot(gs[ax_idx], sharex=signal_axes[0])
            signal_axes.append(ax)
            ax.plot(a_win['time_s'], a_win['diff_forward'], linewidth=0.5, color='purple', label='\u0394 forward: MEMS\u2212GPS (Earth)')
            ax.axhline(0, color='gray', linewidth=0.3)
            ax.set_ylabel('\u0394 fwd\n(m/s\u00b2)')
            ax.legend(fontsize=7, loc='upper right')
            plot_event_markers(ax, events, t_start, t_end)
            ax_idx += 1

            ax = fig.add_subplot(gs[ax_idx], sharex=signal_axes[0])
            signal_axes.append(ax)
            ax.plot(a_win['time_s'], a_win['diff_lateral'], linewidth=0.5, color='olive', label='\u0394 lateral: MEMS\u2212GPS (Earth)')
            ax.axhline(0, color='gray', linewidth=0.3)
            ax.set_ylabel('\u0394 lat\n(m/s\u00b2)')
            ax.legend(fontsize=7, loc='upper right')
            plot_event_markers(ax, events, t_start, t_end)
            ax_idx += 1

        signal_axes[-1].set_xlabel('Time (s)')
        show_time_on_all(signal_axes)
        plt.show()

    interact(
        interactive_zoom,
        time_range=FloatRangeSlider(
            value=[_t_min + _duration * 0.4, _t_min + _duration * 0.4 + 5],
            min=_t_min,
            max=_t_max,
            step=0.5,
            description='Time range:',
            readout_format='.1f',
            layout=Layout(width='80%'),
            style={'description_width': 'initial'},
        ),
    )
else:
    print('No accel data available.')